In [2]:
import requests
import pandas as pd
import time
import os
from pathlib import Path
import boto3, json

if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.config import load_config

CONFIG = load_config()

In [5]:
s3_session = boto3.Session()
s3 = s3_session.client("s3")

import os
from pathlib import Path

EXCLUDE_DIRS = {".git", "__pycache__", ".venv", "node_modules"}
EXCLUDE_FILES = {".env"}

project_root = Path("..").resolve() / "NappeCast"   # depuis notebook/, remonte à la racine du projet

def upload_path(s3_client, path: Path, bucket, s3_prefix, base: Path):
    if not path.exists():
        print(f"⚠️  Introuvable, ignoré : {path}")
        return
    if path.is_file():
        relative = path.relative_to(base).as_posix()
        print(f"Upload : {path} -> s3://{bucket}/{s3_prefix}/{relative}")
        s3_client.upload_file(str(path), bucket, f"{s3_prefix}/{relative}")
    elif path.is_dir():
        for root, dirs, files in os.walk(path):
            dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
            for file in files:
                if file in EXCLUDE_FILES:
                    continue
                local_file = Path(root) / file
                relative = local_file.relative_to(base).as_posix()
                print(f"Upload : {local_file} -> s3://{bucket}/{s3_prefix}/{relative}")
                s3_client.upload_file(str(local_file), bucket, f"{s3_prefix}/{relative}")

# uniquement ce qu'on veut déployer, pas tout le projet ni le dossier notebook/
to_upload = [
    project_root / "docker-compose.yml",
    project_root / "Caddyfile", 
    project_root / "src",
    project_root / "configs",
]

for item in to_upload:
    upload_path(s3, item, "nappecast", "app", base=project_root)

Upload : /home/ronanguilloueee/NappeCast/docker-compose.yml -> s3://nappecast/app/docker-compose.yml
Upload : /home/ronanguilloueee/NappeCast/Caddyfile -> s3://nappecast/app/Caddyfile
Upload : /home/ronanguilloueee/NappeCast/src/config.py -> s3://nappecast/app/src/config.py
Upload : /home/ronanguilloueee/NappeCast/src/__init__.py -> s3://nappecast/app/src/__init__.py
Upload : /home/ronanguilloueee/NappeCast/src/helper/aws.py -> s3://nappecast/app/src/helper/aws.py
Upload : /home/ronanguilloueee/NappeCast/src/helper/data.py -> s3://nappecast/app/src/helper/data.py
Upload : /home/ronanguilloueee/NappeCast/src/helper/__init__.py -> s3://nappecast/app/src/helper/__init__.py
Upload : /home/ronanguilloueee/NappeCast/src/mlflow/docker-compose.yml -> s3://nappecast/app/src/mlflow/docker-compose.yml
Upload : /home/ronanguilloueee/NappeCast/src/mlflow/Dockerfile -> s3://nappecast/app/src/mlflow/Dockerfile
Upload : /home/ronanguilloueee/NappeCast/src/app/app_predictions.py -> s3://nappecast/app/s

bash

ssh -i "nappecast-ec2-key.pem" ec2-user@ec2-51-45-21-195.eu-west-3.compute.amazonaws.com

python3 /home/ec2-user/download_s3.py  
cd /home/ec2-user/app  
sudo docker compose up -d --build  

### Intérroger les evts des images docker
sudo docker compose logs api --tail=50